# 09 - Dashboard data

Owner: F + A

Goal: make the two files the dashboard reads, so the app itself does no cleaning or modelling.

Reads: `PROCESSED / "master.csv"`, `PROCESSED / "predictions_2026.csv"` (from `08_model`, optional) and
`INTERIM / "boundaries.gpkg"`
Writes: `app/artifacts/dashboard.csv` and `app/artifacts/municipalities.geojson`

- `dashboard.csv`: one row per municipality, with every number the app shows, in plain column names
- `municipalities.geojson`: the boundaries, simplified so the map loads quickly on a phone

Both files are saved in GitHub (`app/artifacts/`), because the online dashboard cannot read the Drive. They only
contain public totals.

If `predictions_2026.csv` is missing, the dashboard still works with 2021 turnout and leaves the forecast empty.
Run this notebook again after `08_model` to fill it in.

Run the two setup cells first.

## Setup

In [ ]:
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Thato20-M/democratic-funnel.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/democratic-funnel")
    url = REPO_URL
    try:                                    # private repo: token from Colab Secrets
        from google.colab import userdata
        url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:
        print("No GITHUB_TOKEN in Colab Secrets - cloning will fail for the private repo")
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", url, str(REPO_DIR)], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "src" / "config.py").exists())

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Repo:", REPO_DIR)

In [ ]:
from src import config
from src.config import RAW, INTERIM, PROCESSED, MODELS, FIGURES, SEED
from src.io_utils import read_any, clean_columns, inventory
from src.privacy import drop_personal

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings("ignore", category=FutureWarning)

config.make_dirs(); config.set_seed()
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid"); plt.rcParams["figure.dpi"] = 110
print(config.describe())

## 1. Load

In [ ]:
master = pd.read_csv(PROCESSED / "master.csv")
y11, y16, y21, y26 = (master[master["election"] == y].set_index("muni_code") for y in (2011, 2016, 2021, 2026))

pred_path = PROCESSED / "predictions_2026.csv"
pred = pd.read_csv(pred_path).set_index("muni_code") if pred_path.exists() else None
print("predictions_2026.csv:", "found" if pred is not None else "NOT found - dashboard will use 2021 turnout only")

## 2. One row per municipality

The problem group compares each municipality with the typical (median) municipality, the same way as `07_eda`.
Registration uses the 2026 roll. Turnout uses the 2026 forecast if `08_model` has run, and 2021 turnout if not.

The 2026 forecast is the national-swing forecast from `08_model`: 2021 turnout + the national change the 2024
national election points to, the same for every municipality. We chose it because the model did not beat it on the
2021 test.

For the charts from `07_eda`, the table also has 2021 density, neighbours' turnout, the Census service rates, and
each municipality's registration and turnout against the typical municipality in 2021.

The gap to typical turns this into people: how many more registrations and how many more voters would bring a
municipality up to the typical level.

In [ ]:
d = pd.DataFrame(index=y26.index)
d["Code"] = d.index
d["Municipality"] = y26["muni_name"].where(~y26["muni_name"].str.isupper(), y26["muni_name"].str.title())
d["Province"] = y26["province"]
d["Type"] = y26["muni_category"].map({"A": "Metro", "B": "Local"})

# the funnel
d["Adults who may vote"] = y26["eligible_adults"].round()
d["Registered 2026"] = y26["registered"]
d["Registration rate 2026"] = y26["registered"] / y26["eligible_adults"]
d["Census caution"] = d["Registration rate 2026"] > 1.05          # Census sample too small here - see 07_eda
d["Young adults who may vote"] = y26["eligible_youth"].round()
d["Young adults registered"] = y26["registered_youth"]
d["Youth registration rate"] = y26["registered_youth"] / y26["eligible_youth"]
d["Young adults not registered"] = (y26["eligible_youth"] - y26["registered_youth"]).clip(lower=0).round()

# history
for y, df in [(2011, y11), (2016, y16), (2021, y21)]:
    d[f"Turnout {y}"] = df["t"]
    d[f"Registered {y}"] = df["registered"]
    d[f"Votes cast {y}"] = df["votes_cast"]

# for the EDA charts (07_eda Q5-Q7), 2021
d["Population density"] = y21["pop_density"]                           # people per km2
d["Neighbours' turnout 2021"] = y21["spatial_lag_turnout"]
d["Neighbours' turnout 2016"] = y16["spatial_lag_turnout"]
d["Adults with higher education"] = y21["higher_ed_rate"]
d["Households with piped water"] = y21["piped_water_rate"]
d["Households with electricity"] = y21["electricity_rate"]
d["Households with refuse removal"] = y21["refuse_removal_rate"]
d["Registration vs typical 2021"] = y21["leak_reg"]
d["Turnout vs typical 2021"] = y21["leak_turn"]

# fell much more than the country in 2021: the 15 largest falls against the national swing (average change)
rel_2021 = y21["dt"] - y21["dt"].mean()
d["Worse than expected in 2021"] = d.index.isin(rel_2021.nsmallest(15).index)

# the 2026 forecast: national swing (08_model), optional
if pred is not None:
    d["Predicted turnout 2026"] = pred["t_2026_swing"]
    d["Predicted turnout 2026 (low)"] = pred["t_2026_swing_low"]
    d["Predicted turnout 2026 (high)"] = pred["t_2026_swing_high"]
    d["After COVID (2024)"] = pred["covid_status"]
    d["Predicted turnout 2026 (COVID recovery)"] = pred["t_2026_covid_recovery"]
else:
    for c in ["Predicted turnout 2026", "Predicted turnout 2026 (low)", "Predicted turnout 2026 (high)",
              "After COVID (2024)", "Predicted turnout 2026 (COVID recovery)"]:
        d[c] = np.nan

has_pred = d["Predicted turnout 2026"].notna().all()
d["Turnout used"] = d["Predicted turnout 2026"] if has_pred else d["Turnout 2021"]
d["Turnout basis"] = "predicted 2026" if has_pred else "2021"
print("Turnout used for groups and gaps:", d["Turnout basis"].iloc[0])

In [ ]:
# compare with the typical (median) municipality; registration capped at 100% (Census caution)
r = d["Registration rate 2026"].clip(upper=1.0)
t = d["Turnout used"]
typ_r, typ_t, typ_youth = r.median(), t.median(), d["Youth registration rate"].clip(upper=1.0).median()
d["Typical registration rate"], d["Typical turnout"], d["Typical youth registration rate"] = typ_r, typ_t, typ_youth

reg_gap, turn_gap = np.log(r / typ_r), np.log(t / typ_t)
d["Problem group"] = np.select([(reg_gap < 0) & (turn_gap < 0), reg_gap < 0, turn_gap < 0],
                               ["Both low", "Low registration", "Low turnout"], "Healthy")
d["What to send"] = d["Problem group"].map({
    "Low registration": "Registration drive",
    "Low turnout": "Voter mobilisation",
    "Both low": "Registration drive and voter mobilisation",
    "Healthy": "Nothing urgent"})

# rates and gaps (a gap is the share lost at that step)
d["Registration rate 2021"] = d["Registered 2021"] / d["Adults who may vote"]
d["Participation 2021"] = d["Votes cast 2021"] / d["Adults who may vote"]          # share of ALL adults who voted
d["Registration gap 2026"] = 1 - d["Registration rate 2026"].clip(upper=1.0)       # adults not on the 2026 roll
d["Turnout gap 2021"] = 1 - d["Turnout 2021"]                                    # registered voters who did not vote
d["Participation shortfall 2021"] = 1 - d["Participation 2021"].clip(upper=1.0)   # adults who did not vote
d["Youth registration gap 2026"] = 1 - d["Youth registration rate"].clip(upper=1.0)
d["Adults not registered 2026"] = (d["Adults who may vote"] - d["Registered 2026"]).clip(lower=0).round()
d["Registered who did not vote 2021"] = d["Registered 2021"] - d["Votes cast 2021"]

# out of every 100 adults: registered, and expected to vote
d["Registered per 100 adults"] = (100 * r).round(0)
d["Voters per 100 adults"] = (100 * r * t).round(0)

# gap to typical, in people
d["Registrations needed to reach typical"] = (typ_r * d["Adults who may vote"] - d["Registered 2026"]).clip(lower=0).round(-1)
d["Extra voters needed to reach typical"] = ((typ_t - t) * d["Registered 2026"]).clip(lower=0).round(-1)

# shortfall = how far below typical a municipality is, split into its registration part and turnout part
d["Shortfall: registration part"] = (-reg_gap).clip(lower=0)
d["Shortfall: turnout part"] = (-turn_gap).clip(lower=0)
d["Priority score"] = -(reg_gap + turn_gap)
d["Priority rank"] = d["Priority score"].rank(ascending=False, method="first").astype(int)

d = d.sort_values("Priority rank").reset_index(drop=True)
print(d["Problem group"].value_counts().to_string())
d.head(10)[["Priority rank", "Municipality", "Province", "Problem group", "Registrations needed to reach typical",
            "Extra voters needed to reach typical"]]

## 3. Checks

In [ ]:
assert len(d) == 213 and d["Code"].is_unique
must_be_full = ["Municipality", "Province", "Adults who may vote", "Registered 2026", "Turnout 2021", "Problem group"]
assert d[must_be_full].notna().all().all(), d[must_be_full].isna().sum()
assert d["Turnout used"].between(0.05, 0.95).all()
print("Census caution (registration above 105%):", int(d["Census caution"].sum()), "municipalities")
print("National: registered per 100 adults =",
      round(100 * d["Registered 2026"].sum() / d["Adults who may vote"].sum()))

## 4. Map layer

Simplified to about 1 km of detail, which is enough for a national map and small enough for a phone.

In [ ]:
import geopandas as gpd

ARTIFACTS = REPO_DIR / "app" / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

gdf = gpd.read_file(INTERIM / "boundaries.gpkg")[["muni_code", "geometry"]].to_crs(epsg=4326)
assert set(gdf["muni_code"]) == set(d["Code"]), "boundary codes do not match the data"
gdf["geometry"] = gdf.geometry.simplify(0.008, preserve_topology=True)
geo_path = ARTIFACTS / "municipalities.geojson"
gdf.to_file(geo_path, driver="GeoJSON", COORDINATE_PRECISION=4)
print(f"{geo_path.name}: {geo_path.stat().st_size / 1e6:.1f} MB")

## 5. Save

In [ ]:
out = ARTIFACTS / "dashboard.csv"
d.to_csv(out, index=False)
print(f"{out.name}: {d.shape[0]} rows x {d.shape[1]} columns, {out.stat().st_size / 1e3:.0f} KB")
print(list(d.columns))

### Next

- On your own computer: the files are already in `app/artifacts/`. Commit and push them.
- In Colab: run the cell below to download the two files, then upload them to `app/artifacts/` on GitHub (Add file,
  Upload files).

In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download(str(ARTIFACTS / "dashboard.csv"))
    files.download(str(ARTIFACTS / "municipalities.geojson"))